# Debug GUI Notebook
This notebook isolates the GUI logic to debug widget responsiveness, state management, and the 'Upload Zip' functionality.
Backend functions (`setup_env`, `start_training`, etc.) are mocked.

In [1]:
import time
import os
import ipywidgets as widgets
from IPython.display import display

# === Mock Globals & Functions ===

out = widgets.Output()
out_viewer = widgets.Output()

def log(msg):
    with out:
        print(f"[LOG] {msg}")

def run_command(cmd):
    log(f"[CMD] {cmd}")
    time.sleep(0.5)

def setup_env(b):
    log("Mock Setup Env")
    time.sleep(1)

def install_deps(b):
    log("Mock Install Deps")
    time.sleep(1)

def prepare_data(b):
    log("Mock Prepare Data")
    if 'file_upload_widget' in globals() and file_upload_widget.value:
         log(f"Upload widget has content (len={len(file_upload_widget.value)})")
         # Iterate to show structure
         val = file_upload_widget.value
         if isinstance(val, dict):
             for name, info in val.items():
                 log(f" - File: {name}, Size: {len(info['content'])}")
         elif isinstance(val, (list, tuple)):
             for item in val:
                 log(f" - File: {item['name']}, Size: {len(item['content'])}")
    else:
         log("Upload widget empty or not found.")
    time.sleep(1)

def start_training(b):
    log("Mock Start Training")
    time.sleep(1)

def start_viewer(b):
    log("Mock Start Viewer")
    time.sleep(1)

# Tailscale Mock
class MockTailscale:
    def __init__(self):
        self.connected = False
        self.status_callback = None
    def toggle(self, change):
        self.connected = change['new']
        status = "Connected" if self.connected else "Disconnected"
        if self.status_callback: self.status_callback(status)
        log(f"Tailscale toggled: {status}")
tailscale_conn = MockTailscale()

# Global Variables Mock
Source_Path = "/content/data"
Output_Path = "/content/output"
Iterations = 7000
SH_Degree = 3
White_Background = False
Eval_Mode = False
DryRun = False
print("Mocks initialized.")

Mocks initialized.


In [2]:
class GUIWidgets:
    def __init__(self):
        self.style = {'description_width': 'initial'}
        self.layout = widgets.Layout(width='auto')
        
        # --- Status Indicator ---
        self.status_html = widgets.HTML(
            value='''
            <style>
            .status-box { padding: 5px; border-radius: 4px; font-weight: bold; }
            .status-ready { background-color: #e6ffed; color: #2da44e; border: 1px solid #2da44e; }
            .status-busy { background-color: #fff8c5; color: #bf8b2d; border: 1px solid #bf8b2d; }
            .loader { border: 3px solid #f3f3f3; border-top: 3px solid #3498db; border-radius: 50%; width: 14px; height: 14px; animation: spin 1s linear infinite; display: inline-block; vertical-align: middle; margin-right: 5px; }
            @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
            </style>
            <div class="status-box status-ready">✅ Ready</div>
            '''
        )

        # --- Core Widgets ---
        self.lbl_tailscale_status = widgets.Label(value="Disconnected", style={'text_color': 'gray'})
        self.cb_tailscale = widgets.Checkbox(value=False, description='Connect Tailscale (ssh)', style=self.style)
        self.cb_dryrun = widgets.Checkbox(value=False, description='Dry Run (No GPU)', style=self.style)
        
        self.btn_env = widgets.Button(description="1. Setup Environment", button_style='primary', layout=self.layout)
        self.dd_rasterizer = widgets.Dropdown(options=['Standard', 'Accelerated (Sparse Adam)'], value='Standard', description='Rasterizer:', style=self.style)
        self.btn_deps = widgets.Button(description="2. Install Dependencies", button_style='info', layout=self.layout)
        
        self.dd_datasource = widgets.Dropdown(options=['Demo Data', 'Google Drive', 'Upload Zip', 'Custom URL', 'Local Folder'], value='Demo Data', description='Data Source:', style=self.style)
        self.txt_source_path = widgets.Text(value='', placeholder='Drive Path / URL / Folder Path', description='Path / URL:', style=self.style)
        self.btn_browse = widgets.Button(description="📂 Browse", layout=widgets.Layout(width='100px'))
        self.file_upload = widgets.FileUpload(accept='.zip', multiple=False)
        self.file_upload.layout.display = 'none'
        self.lbl_upload_instruction = widgets.HTML("<i>Select 'Upload Zip' and click <b>'3. Prepare Data'</b> to launch the upload.</i>")
        self.lbl_upload_instruction.layout.display = 'none'
        
        # File Browser Widgets (Source Path)
        self.lbl_browser_title = widgets.Label(value="Select File", style={'font_weight': 'bold'})
        self.lbl_path = widgets.Label(f"Current: /")
        self.sel_files = widgets.Select(options=[], rows=10, layout=widgets.Layout(width='100%'))
        self.btn_up = widgets.Button(description="⬆ Up", layout=widgets.Layout(width='80px'))
        self.btn_select = widgets.Button(description="Select", button_style='primary', layout=widgets.Layout(width='80px'))
        self.btn_cancel_browser = widgets.Button(description="Cancel", layout=widgets.Layout(width='80px'))
        
        self.browser_box = widgets.VBox([
            self.lbl_browser_title,
            widgets.HBox([self.btn_up, self.lbl_path]),
            self.sel_files,
            widgets.HBox([self.btn_cancel_browser, self.btn_select])
        ])
        self.browser_box.layout.display = 'none'

        # File Browser Widgets (Viewer) - DUPLICATED
        self.lbl_browser_title_v = widgets.Label(value="Select Viewer File", style={'font_weight': 'bold'})
        self.lbl_path_v = widgets.Label(f"Current: /")
        self.sel_files_v = widgets.Select(options=[], rows=10, layout=widgets.Layout(width='100%'))
        self.btn_up_v = widgets.Button(description="⬆ Up", layout=widgets.Layout(width='80px'))
        self.btn_select_v = widgets.Button(description="Select", button_style='primary', layout=widgets.Layout(width='80px'))
        self.btn_cancel_browser_v = widgets.Button(description="Cancel", layout=widgets.Layout(width='80px'))
        
        self.browser_box_viewer = widgets.VBox([
            self.lbl_browser_title_v,
            widgets.HBox([self.btn_up_v, self.lbl_path_v]),
            self.sel_files_v,
            widgets.HBox([self.btn_cancel_browser_v, self.btn_select_v])
        ])
        self.browser_box_viewer.layout.display = 'none'

        self.btn_data = widgets.Button(description="3. Prepare Data", button_style='warning', layout=self.layout)
        
        # Training Options
        self.cb_antialiasing = widgets.Checkbox(value=False, description='Anti-aliasing', style=self.style)
        self.cb_exposure = widgets.Checkbox(value=False, description='Exposure Comp', style=self.style)
        self.cb_depth = widgets.Checkbox(value=False, description='Depth Reg', style=self.style)
        self.txt_depth_path = widgets.Text(value='', placeholder='Depth maps path', description='Depth Path:', display='none', style=self.style)
        self.txt_depth_path.layout.display = 'none'
        self.cb_sparse_adam = widgets.Checkbox(value=False, description='Sparse Adam', disabled=True, style=self.style)
        
        self.btn_train = widgets.Button(description="4. Train", button_style='success', layout=self.layout)
        
        # Viewer Widgets
        self.txt_viewer_path = widgets.Text(value='', placeholder='Path to .ply (leave empty to auto-detect)', description='Viewer File:', style=self.style)
        self.cb_embedded = widgets.Checkbox(value=True, description='Embed Viewer', style=self.style)
        self.btn_browse_viewer = widgets.Button(description="📂", layout=widgets.Layout(width='40px'))
        
        self.btn_view = widgets.Button(description="5. Start Viewer", button_style='danger', layout=self.layout)
        self.btn_stop_viewer = widgets.Button(description="Stop Viewer", button_style='warning', icon='stop', layout=widgets.Layout(width='auto'))

        # --- Layout Container ---
        self.container = widgets.VBox([
            widgets.HBox([widgets.HTML("<h3>Gaussian Splatting Controller</h3>"), self.status_html]),
            widgets.HBox([self.cb_tailscale, self.lbl_tailscale_status]),
            widgets.HBox([self.btn_env]),
            widgets.HBox([self.dd_rasterizer, self.btn_deps]),
            widgets.HTML("<hr>"),
            widgets.HBox([self.dd_datasource, self.txt_source_path, self.btn_browse]),
            widgets.HBox([self.file_upload, self.lbl_upload_instruction]),
            self.browser_box,
            widgets.HBox([self.btn_data]),
            widgets.HTML("<hr>"),
            widgets.Label("Training Options (New Features):"),
            widgets.HBox([self.cb_antialiasing, self.cb_exposure, self.cb_sparse_adam]),
            widgets.HBox([self.cb_depth, self.txt_depth_path]),
            widgets.HTML("<br>"),
            widgets.HBox([self.cb_dryrun]),
            
            widgets.HTML("<hr>"),
            widgets.Label("Viewer Config:"),
            widgets.HBox([self.txt_viewer_path, self.btn_browse_viewer]),
            widgets.HBox([self.cb_embedded]),
            self.browser_box_viewer,
            widgets.HBox([self.btn_train]),
            widgets.HTML("<br>"),
            widgets.HBox([self.btn_view, self.btn_stop_viewer]),
        
            out_viewer,
            widgets.Label("Logs:"),
            out  # Using global 'out' widget
        ])

    def set_status(self, state, msg):
        if state == 'busy':
            self.status_html.value = f'''
            <style>.status-box {{ padding: 5px; border-radius: 4px; font-weight: bold; }} .status-busy {{ background-color: #fff8c5; color: #bf8b2d; border: 1px solid #bf8b2d; }} .loader {{ border: 3px solid #f3f3f3; border-top: 3px solid #3498db; border-radius: 50%; width: 14px; height: 14px; animation: spin 1s linear infinite; display: inline-block; vertical-align: middle; margin-right: 5px; }} @keyframes spin {{ 0% {{ transform: rotate(0deg); }} 100% {{ transform: rotate(360deg); }} }}</style>
            <div class="status-box status-busy"><div class="loader"></div> {msg}</div>'''
        else:
            self.status_html.value = '''
             <style>.status-box {{ padding: 5px; border-radius: 4px; font-weight: bold; }} .status-ready {{ background-color: #e6ffed; color: #2da44e; border: 1px solid #2da44e; }}</style>
            <div class="status-box status-ready">✅ Ready</div>'''

        # Lockable widgets during async actions
        # Exclude Tailscale from this list to keep it independent
        self.lockable_widgets = [
            self.cb_dryrun, self.btn_env, self.dd_rasterizer,
            self.btn_deps, self.dd_datasource, self.txt_source_path, self.btn_browse,
            self.btn_data, self.cb_antialiasing, self.cb_exposure,
            self.cb_depth, self.txt_depth_path, self.btn_train, self.btn_view, self.txt_viewer_path, self.btn_browse_viewer, self.cb_embedded,
            self.btn_stop_viewer
        ]


In [3]:
class GUIController:
    def __init__(self, view):
        self.view = view
        self.current_path = os.getcwd()
        self.current_path_v = os.getcwd() # Separate path state for viewer browser
        self.active_input = None
        
        # Init browsers
        self._update_browser(self.view.sel_files, self.view.lbl_path, self.current_path)
        self._update_browser(self.view.sel_files_v, self.view.lbl_path_v, self.current_path_v)

        # Bind events last to ensure all methods are defined
        self._bind_events()

    def _bind_events(self):
        # Tailscale
        if 'tailscale_conn' in globals():
            self.view.cb_tailscale.observe(self._on_tailscale_change, names='value')
            if hasattr(tailscale_conn, 'status_callback'):
                 tailscale_conn.status_callback = lambda msg: setattr(self.view.lbl_tailscale_status, 'value', msg)
        
        # Visibility Logic
        self.view.dd_rasterizer.observe(self._on_rasterizer_change, names='value')
        self.view.cb_depth.observe(self._on_depth_change, names='value')
        self.view.dd_datasource.observe(self._on_datasource_change, names='value')
        
        # File Browser 1 (Source)
        self.view.btn_browse.on_click(lambda b: self._open_browser_source(self.view.txt_source_path, "Source Path"))
        self.view.btn_up.on_click(lambda b: self._on_up(self.view.sel_files, self.view.lbl_path, is_viewer=False))
        self.view.sel_files.observe(lambda change: self._on_select_item(change, self.view.sel_files, self.view.lbl_path, is_viewer=False), names='value')
        self.view.btn_select.on_click(lambda b: self._on_confirm_select(self.view.browser_box, self.view.sel_files, is_viewer=False))
        self.view.btn_cancel_browser.on_click(lambda b: self._on_cancel_browser(self.view.browser_box))

        # File Browser 2 (Viewer)
        self.view.btn_browse_viewer.on_click(lambda b: self._open_browser_viewer(self.view.txt_viewer_path, "Viewer File"))
        self.view.btn_up_v.on_click(lambda b: self._on_up(self.view.sel_files_v, self.view.lbl_path_v, is_viewer=True))
        self.view.sel_files_v.observe(lambda change: self._on_select_item(change, self.view.sel_files_v, self.view.lbl_path_v, is_viewer=True), names='value')
        self.view.btn_select_v.on_click(lambda b: self._on_confirm_select(self.view.browser_box_viewer, self.view.sel_files_v, is_viewer=True))
        self.view.btn_cancel_browser_v.on_click(lambda b: self._on_cancel_browser(self.view.browser_box_viewer))
        
        # Actions with Async Feedback
        self.view.btn_env.on_click(lambda b: self._run_action("Setting up Env...", setup_env))
        self.view.btn_deps.on_click(lambda b: self._run_action("Installing Deps...", install_deps))
        self.view.btn_data.on_click(lambda b: self._run_action("Preparing Data...", prepare_data))
        self.view.btn_train.on_click(lambda b: self._run_action("Training...", start_training))
        self.view.btn_view.on_click(lambda b: self._run_action("Starting Viewer...", start_viewer))
        self.view.btn_stop_viewer.on_click(self._on_stop_viewer)

    def _run_action(self, msg, func):
        self.view.set_status('busy', msg)
        for w in self.view.lockable_widgets:
            w.disabled = True

        try:
            func(None)
        except Exception as e:
            log(f"Error: {e}")
        finally:
            self.view.set_status('ready', "")
            for w in self.view.lockable_widgets:
                w.disabled = False
    
    def _on_stop_viewer(self, b):
        self.view.set_status('busy', "Stopping Viewer...")
        try:
             # Kill port 8000
             # We use fuser to kill the process on port 8000
             run_command(f"fuser -k 8000/tcp")
             log("Viewer stopped.")
        except Exception as e:
             log(f"Error stopping viewer: {e}")
        finally:
             self.view.set_status('ready', "")

    def _on_tailscale_change(self, change):
        if 'tailscale_conn' in globals():
            tailscale_conn.toggle(change)

    # --- Event Handlers ---
    def _on_rasterizer_change(self, change):
        if change['new'].startswith('Accelerated'):
            self.view.cb_sparse_adam.value = True
        else:
            self.view.cb_sparse_adam.value = False

    def _on_depth_change(self, change):
        if change['new']:
            self.view.txt_depth_path.layout.display = 'flex'
        else:
            self.view.txt_depth_path.layout.display = 'none'

    def _on_datasource_change(self, change):
        val = change['new']
        if val in ['Google Drive', 'Local Folder']:
            self.view.txt_source_path.layout.display = 'flex'
            self.view.btn_browse.layout.display = 'block'
            self.view.lbl_upload_instruction.layout.display = 'none'
        elif val == 'Upload Zip':
            self.view.txt_source_path.layout.display = 'none'
            self.view.btn_browse.layout.display = 'none'
            self.view.file_upload.layout.display = 'block'
            self.view.lbl_upload_instruction.layout.display = 'block'
        elif val == 'Custom URL':
            self.view.txt_source_path.layout.display = 'flex'
            self.view.btn_browse.layout.display = 'none'
            self.view.file_upload.layout.display = 'none'
            self.view.lbl_upload_instruction.layout.display = 'none'
        else: # Demo Data
            self.view.txt_source_path.layout.display = 'none'
            self.view.btn_browse.layout.display = 'none'
            self.view.file_upload.layout.display = 'none'
            self.view.lbl_upload_instruction.layout.display = 'none'

    # --- Browser Logic ---
    def _update_browser(self, widget_sel, widget_lbl, path):
        try:
            if not os.path.exists(path): path = '/content'
            items = sorted(os.listdir(path))
            formatted_items = []
            for item in items:
                if os.path.isdir(os.path.join(path, item)):
                    formatted_items.append(f"\ud83d\udcc2 {item}")
                else:
                    formatted_items.append(f"\ud83d\udcc4 {item}")
            widget_sel.options = formatted_items
            widget_lbl.value = f"Current: {path}"
        except Exception as e:
            widget_lbl.value = f"Error: {e}"

    def _on_up(self, widget_sel, widget_lbl, is_viewer=False):
        if is_viewer:
             self.current_path_v = os.path.dirname(self.current_path_v)
             self._update_browser(widget_sel, widget_lbl, self.current_path_v)
        else:
             self.current_path = os.path.dirname(self.current_path)
             self._update_browser(widget_sel, widget_lbl, self.current_path)

    def _on_select_item(self, change, widget_sel, widget_lbl, is_viewer=False):
        if change['new']:
            name = change['new'].split(' ', 1)[1]
            if is_viewer:
                full_path = os.path.join(self.current_path_v, name)
                if os.path.isdir(full_path):
                     self.current_path_v = full_path
                     self._update_browser(widget_sel, widget_lbl, self.current_path_v)
            else:
                full_path = os.path.join(self.current_path, name)
                if os.path.isdir(full_path):
                     self.current_path = full_path
                     self._update_browser(widget_sel, widget_lbl, self.current_path)

    def _on_confirm_select(self, browse_box_widget, widget_sel, is_viewer=False):
        val = widget_sel.value
        path_to_use = self.current_path_v if is_viewer else self.current_path
        
        if val:
            name = val.split(' ', 1)[1]
            path_to_use = os.path.join(path_to_use, name)
        
        if self.active_input:
            self.active_input.value = path_to_use
        
        browse_box_widget.layout.display = 'none'
        self.active_input = None

    def _open_browser_source(self, target_widget, title="Select File"):
        self.active_input = target_widget
        self.view.lbl_browser_title.value = f"Selecting: {title}"
        self.view.browser_box.layout.display = 'block'
        self.view.browser_box_viewer.layout.display = 'none' # Ensure other is closed
        self._update_browser(self.view.sel_files, self.view.lbl_path, self.current_path)

    def _open_browser_viewer(self, target_widget, title="Select Viewer File"):
        self.active_input = target_widget
        self.view.lbl_browser_title_v.value = f"Selecting: {title}"
        self.view.browser_box_viewer.layout.display = 'block'
        self.view.browser_box.layout.display = 'none' # Ensure other is closed
        self._update_browser(self.view.sel_files_v, self.view.lbl_path_v, self.current_path_v)

    def _on_cancel_browser(self, browse_box_widget):
        browse_box_widget.layout.display = 'none'
        self.active_input = None


In [4]:
if __name__ == "__main__":
    # 1. Instantiate View
    view = GUIWidgets()
    
    # 2. Restore logic
    if 'tailscale_conn' in globals() and tailscale_conn.connected:
        view.cb_tailscale.value = True

    # Export widgets
    globals()['file_upload_widget'] = view.file_upload
    globals()['cb_tailscale'] = view.cb_tailscale
    globals()['cb_dryrun'] = view.cb_dryrun
    globals()['dd_datasource'] = view.dd_datasource
    globals()['txt_source_path'] = view.txt_source_path
    globals()['dd_rasterizer'] = view.dd_rasterizer
    globals()['cb_antialiasing'] = view.cb_antialiasing
    globals()['cb_exposure'] = view.cb_exposure
    globals()['cb_depth'] = view.cb_depth
    globals()['txt_depth_path'] = view.txt_depth_path
    globals()['cb_sparse_adam'] = view.cb_sparse_adam
    globals()['cb_embedded'] = view.cb_embedded
    
    # 3. Instantiate Controller
    controller = GUIController(view)
    
    # 4. Initialize State
    controller._on_datasource_change({'new': view.dd_datasource.value})
    
    # 5. Display GUI
    display(view.container)
